<a href="https://colab.research.google.com/github/0MSA0/supreme-robot/blob/Sebastian_Week_2/attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Basic Attention Unit

In [4]:
import torch.nn as nn
import torch

class BasicAttention(nn.Module):
  def __init__(self, d_in: int, d_out: int, qkv_bias = False):
    super().__init__()
    self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

  def forward(self, x):
    keys = self.W_key(x)
    queries = self.W_query(x)
    values = self.W_value(x)

    attention_scores = queries @ keys.T
    attention_weights = torch.softmax(attention_scores / keys.shape[-1]**0.5, dim=-1)
    context_vector = attention_weights @ values

    return context_vector

# Causal Attention with dropout

In [6]:
class CausalAttention(nn.Module):
  def __init__(self, d_in: int, d_out: int, context_length: int, dropout: float, qkv_bias = False):
    super().__init__()
    self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.dropout = nn.Dropout(dropout)
    # Create mask with -inf above the diagonal
    self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

  def forward(self, x):
    b, num_tokens, d_in = x.shape
    keys = self.W_key(x)
    queries = self.W_query(x)
    values = self.W_value(x)

    attention_scores = queries @ keys.transpose(1, 2)

    # Mask attention scores
    masked_att_scores = attention_scores.masked_fill(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)

    # Get attention weights using softmax
    attention_weights = torch.softmax(masked_att_scores / keys.shape[-1]**0.5, dim=-1)

    # Apply dropout
    attention_weights = self.dropout(attention_weights)

    context_vector = attention_weights @ values

    return context_vector

# Multi head attention

In [21]:
class MultiHeadAttentionWrapper(nn.Module):
  def __init__(self, d_in: int, d_out: int, context_length: int, dropout: float, num_heads: int, qkv_bias = False):
    super().__init__()

    # Create multiple attention units
    self.heads = nn.ModuleList(
        [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias) for _ in range(num_heads)]
    )
    self.out_proj = nn.Linear(d_out * num_heads, d_out * num_heads)

  def forward(self, x):
    context_vec = torch.cat([head(x) for head in self.heads], dim=-1)
    return self.out_proj(context_vec)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()

        assert (d_out % num_heads == 0), "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        # Reduce the projection dim to match desired output dim
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        # Linear layer to combine head outputs
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # Add num_heads dimension
        # (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Same as causal attention
        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1, 2)
        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

# Testing

In [23]:
if __name__ == "__main__":
  inputs = torch.tensor(
    [[0.43, 0.15, 0.89], # Your     (x^1)
    [0.55, 0.87, 0.66], # journey  (x^2)
    [0.57, 0.85, 0.64], # starts   (x^3)
    [0.22, 0.58, 0.33], # with     (x^4)
    [0.77, 0.25, 0.10], # one      (x^5)
    [0.05, 0.80, 0.55]] # step     (x^6)
  )
  d_in = inputs.shape[1] # the input embedding size, d=3
  d_out = 2 # the output embedding size, d=2
  batch = torch.stack((inputs, inputs), dim=0)
  context_length = batch.shape[1]

  # Basic attention
  torch.manual_seed(789)
  ba_cv = BasicAttention(d_in, d_out)
  print("ba context_vecs: ", ba_cv(inputs))

  # Causal attention
  torch.manual_seed(123)
  ca_cv = CausalAttention(d_in, d_out, context_length, 0.0)
  print("ca context_vecs: ", ca_cv(batch))

  # Multi head attention
  torch.manual_seed(123)
  batch_size, context_length, d_in = batch.shape
  d_out = 2
  mhaw = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, num_heads=2)

  mhaw_cv = mhaw(batch)
  print("mhaw context_vecs:", mhaw_cv)
  print("mhaw context_vecs.shape:", mhaw_cv.shape)

  torch.manual_seed(123)
  mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)
  mha_cv = mha(batch)
  print("mha context_vecs:", mha_cv)
  print("mha context_vecs.shape:", mha_cv.shape)



ba context_vecs:  tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)
ca context_vecs:  tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]],

        [[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>)
mhaw context_vecs: tensor([[[ 0.1127,  0.2046, -0.0721, -0.5020],
         [ 0.1688,  0.3739, -0.0617, -0.5847],
         [ 0.1877,  0.4236, -0.0588, -0.6127],
         [ 0.1889,  0.4139, -0.0581, -0.6071],
         [ 0.1896,  0.4092, -0.0589, -0.6092],
         [ 0.1903,  0.4140, -0.0576, -0.6058]],

        [[ 0.1127,  0.2046, -0.0721, -0.5020],
         [ 0.1688,  0.3739, -0.0617, -0